# 04 – Feature Engineering: Variables de Ingresos

**Proyecto:** Predicción de Subempleo por Insuficiencia de Horas — EPEN 2024  
**Etapa:** Feature Engineering – Paso 4 de 4  
**Dataset de entrada:** `data/feature_engineering/epen_fe_employment.csv`  
**Dataset de salida:** `data/feature_engineering/epen_features_final.csv`

## Objetivo

Crear variables derivadas del **nivel de ingresos** del trabajador. Es el último paso del feature engineering antes del train/test split.

El dataset resultante `epen_features_final.csv` contiene **todas las features** (demográficas, educativas, laborales e ingresos) listas para el pipeline de modelado.

### Variables base utilizadas
| Variable | Descripción |
|:--------:|:------------|
| INGTOT | Ingreso total del trabajador |
| INGTOTP | Ingreso total del hogar per cápita |
| ingtrabw | Ingreso laboral semanal |
| I339_1 | Ingreso por empleo principal |
| I342 | Ingreso por empleo secundario |
| I345_1 | Otro ingreso 1 |
| I348 | Otro ingreso 2 |
| INGTOT_log | Log del ingreso total (ya calculado en preprocesamiento) |
| INGTOTP_log | Log del ingreso per cápita |
| INGTRABW_log | Log del ingreso laboral |
| INGTOT_outlier_flag | Flag de outlier en ingreso total |
| INGTRABW_outlier_flag | Flag de outlier en ingreso laboral |

### Restricciones
- No se usa: `P209H`, `C333`, `C334`, `fa_son24`
- No se realiza train/test split en este notebook
- Este es el **último** notebook de feature engineering; el siguiente paso es `train_test_split.ipynb`

---
## 1. Cargar Librerías

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Librerías cargadas correctamente.')

Librerías cargadas correctamente.


---
## 2. Cargar Dataset

In [2]:
INPUT_PATH = Path('../data/feature_engineering/epen_fe_employment.csv')

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f'No se encontró el archivo de entrada: {INPUT_PATH}\n'
        'Ejecuta primero: 04_feature_engineering/03_employment_features.ipynb'
    )

df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f'Dataset cargado : {INPUT_PATH.name}')
print(f'Dimensiones     : {df.shape[0]:,} filas x {df.shape[1]} columnas')

Dataset cargado : epen_fe_employment.csv
Dimensiones     : 24,054 filas x 98 columnas


---
## 3. Validación Inicial

In [3]:
assert 'target_subempleo_horas' in df.columns, \
    "ERROR: 'target_subempleo_horas' no encontrado."
assert df['target_subempleo_horas'].isnull().sum() == 0, \
    'ERROR: target_subempleo_horas contiene nulos.'
print('target_subempleo_horas presente y sin nulos: OK')

for lv in ['P209H', 'C333', 'C334']:
    assert lv not in df.columns, f'ERROR: variable de leakage {lv} encontrada.'
print('Variables de leakage ausentes: OK')

# Verificar horas_totales (necesaria para ingreso por hora)
if 'horas_totales' not in df.columns:
    print('ADVERTENCIA: horas_totales no encontrada. ingreso_por_hora no podrá calcularse.')
else:
    print('horas_totales presente: OK')

print('\nDistribución del target:')
counts = df['target_subempleo_horas'].value_counts()
pct    = df['target_subempleo_horas'].value_counts(normalize=True) * 100
display(pd.DataFrame({'conteo': counts, 'porcentaje (%)': pct.round(2)}))

target_subempleo_horas presente y sin nulos: OK
Variables de leakage ausentes: OK
horas_totales presente: OK

Distribución del target:


,conteo,porcentaje (%)
target_subempleo_horas,,
0,18064,75.1000
1,5990,24.9000


---
## 4. Crear Copia de Trabajo

In [4]:
df_fe = df.copy()
n_original = df_fe.shape[0]
features_created = []

print(f'Copia creada: {df_fe.shape[0]:,} filas x {df_fe.shape[1]} columnas')

Copia creada: 24,054 filas x 98 columnas


---
## 5. Ingresos Base

Estandariza las variables de ingreso con nombres descriptivos.

**Features creados:** `ingreso_total`, `ingreso_principal`, `ingreso_laboral`

In [5]:
# Ingreso total
if 'INGTOT' in df_fe.columns:
    df_fe['ingreso_total'] = pd.to_numeric(df_fe['INGTOT'], errors='coerce')
    features_created.append('ingreso_total')
    print(f"ingreso_total  — media: {df_fe['ingreso_total'].mean():,.2f}  | nulos: {df_fe['ingreso_total'].isnull().sum()}")
else:
    print('ADVERTENCIA: INGTOT no encontrado.')

# Ingreso per cápita del hogar
if 'INGTOTP' in df_fe.columns:
    df_fe['ingreso_principal'] = pd.to_numeric(df_fe['INGTOTP'], errors='coerce')
    features_created.append('ingreso_principal')
    print(f"ingreso_principal (INGTOTP) — media: {df_fe['ingreso_principal'].mean():,.2f}")
elif 'I339_1' in df_fe.columns:
    df_fe['ingreso_principal'] = pd.to_numeric(df_fe['I339_1'], errors='coerce')
    features_created.append('ingreso_principal')
    print(f"ingreso_principal (I339_1) — media: {df_fe['ingreso_principal'].mean():,.2f}")
else:
    print('ADVERTENCIA: INGTOTP e I339_1 no encontrados. ingreso_principal no creado.')

# Ingreso laboral (trabajo)
if 'ingtrabw' in df_fe.columns:
    df_fe['ingreso_laboral'] = pd.to_numeric(df_fe['ingtrabw'], errors='coerce')
    features_created.append('ingreso_laboral')
    print(f"ingreso_laboral (ingtrabw) — media: {df_fe['ingreso_laboral'].mean():,.2f}")
else:
    print('ADVERTENCIA: ingtrabw no encontrado. ingreso_laboral no creado.')

ingreso_total  — media: 2,047.38  | nulos: 774
ingreso_principal (INGTOTP) — media: 1,976.75
ingreso_laboral (ingtrabw) — media: 2,195.83


---
## 6. Ingresos en Escala Logarítmica

Usa las versiones log ya calculadas en la etapa de preprocesamiento cuando están disponibles.
Si no están disponibles, se calculan en este notebook.

**Features creados:** `ingreso_total_log`, `ingreso_principal_log`, `ingreso_laboral_log`

In [6]:
def safe_log(series):
    """Aplica log1p si los valores son validos (>= 0), sino NaN."""
    s = pd.to_numeric(series, errors='coerce')
    return np.where(s >= 0, np.log1p(s), np.nan)

# Log de ingreso total
if 'INGTOT_log' in df_fe.columns:
    df_fe['ingreso_total_log'] = pd.to_numeric(df_fe['INGTOT_log'], errors='coerce')
    print('ingreso_total_log: desde INGTOT_log (preprocesamiento)')
elif 'ingreso_total' in df_fe.columns:
    df_fe['ingreso_total_log'] = safe_log(df_fe['ingreso_total'])
    print('ingreso_total_log: calculado como log1p(ingreso_total)')
else:
    print('ADVERTENCIA: ingreso_total_log no pudo crearse.')

if 'ingreso_total_log' in df_fe.columns:
    features_created.append('ingreso_total_log')

# Log de ingreso principal
if 'INGTOTP_log' in df_fe.columns:
    df_fe['ingreso_principal_log'] = pd.to_numeric(df_fe['INGTOTP_log'], errors='coerce')
    print('ingreso_principal_log: desde INGTOTP_log')
elif 'ingreso_principal' in df_fe.columns:
    df_fe['ingreso_principal_log'] = safe_log(df_fe['ingreso_principal'])
    print('ingreso_principal_log: calculado como log1p(ingreso_principal)')
else:
    print('ADVERTENCIA: ingreso_principal_log no pudo crearse.')

if 'ingreso_principal_log' in df_fe.columns:
    features_created.append('ingreso_principal_log')

# Log de ingreso laboral
if 'INGTRABW_log' in df_fe.columns:
    df_fe['ingreso_laboral_log'] = pd.to_numeric(df_fe['INGTRABW_log'], errors='coerce')
    print('ingreso_laboral_log: desde INGTRABW_log')
elif 'ingreso_laboral' in df_fe.columns:
    df_fe['ingreso_laboral_log'] = safe_log(df_fe['ingreso_laboral'])
    print('ingreso_laboral_log: calculado como log1p(ingreso_laboral)')
else:
    print('ADVERTENCIA: ingreso_laboral_log no pudo crearse.')

if 'ingreso_laboral_log' in df_fe.columns:
    features_created.append('ingreso_laboral_log')

ingreso_total_log: desde INGTOT_log (preprocesamiento)
ingreso_principal_log: desde INGTOTP_log
ingreso_laboral_log: desde INGTRABW_log


---
## 7. Ingreso por Hora

Ingreso laboral mensual dividido por las horas trabajadas en el mes (estimado como horas semanales × 4.33 semanas/mes).

**Features creados:** `ingreso_por_hora`, `ingreso_por_hora_log`

In [7]:
if 'ingreso_laboral' in df_fe.columns and 'horas_totales' in df_fe.columns:
    horas_mes = df_fe['horas_totales'] * 4.33
    # Evitar división por cero
    df_fe['ingreso_por_hora'] = np.where(
        horas_mes > 0,
        df_fe['ingreso_laboral'] / horas_mes,
        np.nan
    )
    df_fe['ingreso_por_hora_log'] = safe_log(df_fe['ingreso_por_hora'])
    features_created += ['ingreso_por_hora', 'ingreso_por_hora_log']

    print(f"ingreso_por_hora — media: {df_fe['ingreso_por_hora'].mean():,.2f}")
    print(f"ingreso_por_hora — nulos: {df_fe['ingreso_por_hora'].isnull().sum()}")
else:
    print('ADVERTENCIA: ingreso_laboral o horas_totales no disponibles. ingreso_por_hora no creado.')

ingreso_por_hora — media: 12.32
ingreso_por_hora — nulos: 774


---
## 8. Ingreso Secundario y No Laboral

**Variables base:** `I345_1`, `I348`

**Features creados:** `ingreso_secundario_total`, `tiene_ingreso_secundario`

In [8]:
ing_sec_cols = [c for c in ['I345_1', 'I348', 'I342'] if c in df_fe.columns]

if ing_sec_cols:
    ing_sec_df = df_fe[ing_sec_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
    df_fe['ingreso_secundario_total'] = ing_sec_df.sum(axis=1)
    df_fe['tiene_ingreso_secundario'] = (df_fe['ingreso_secundario_total'] > 0).astype(int)
    features_created += ['ingreso_secundario_total', 'tiene_ingreso_secundario']

    print(f'Columnas usadas: {ing_sec_cols}')
    print(f"ingreso_secundario_total — media: {df_fe['ingreso_secundario_total'].mean():,.2f}")
    print(f"tiene_ingreso_secundario : {df_fe['tiene_ingreso_secundario'].sum():,}  ({df_fe['tiene_ingreso_secundario'].mean()*100:.1f}%)")
else:
    print('ADVERTENCIA: I345_1, I348 e I342 no encontrados.')

Columnas usadas: ['I345_1', 'I348', 'I342']
ingreso_secundario_total — media: 616.69
tiene_ingreso_secundario : 9,683  (40.3%)


---
## 9. Proporciones e Indicadores Relativos de Ingreso

**Features creados:**
- `proporcion_ingreso_principal`: ingreso_principal / ingreso_total
- `dependencia_ingreso_principal`: 1 si proporcion_ingreso_principal >= 0.8
- `bajo_ingreso_relativo`: 1 si ingreso_total < percentil 25
- `alto_ingreso_relativo`: 1 si ingreso_total > percentil 75
- `ingreso_cero`: 1 si ingreso_total == 0

In [9]:
if 'ingreso_total' in df_fe.columns and 'ingreso_principal' in df_fe.columns:
    # Proporción de ingreso principal sobre total
    df_fe['proporcion_ingreso_principal'] = np.where(
        df_fe['ingreso_total'] > 0,
        df_fe['ingreso_principal'] / df_fe['ingreso_total'],
        np.nan
    )
    df_fe['dependencia_ingreso_principal'] = (
        df_fe['proporcion_ingreso_principal'] >= 0.8
    ).astype(int)
    features_created += ['proporcion_ingreso_principal', 'dependencia_ingreso_principal']
    print(f"proporcion_ingreso_principal — media: {df_fe['proporcion_ingreso_principal'].mean():.4f}")

if 'ingreso_total' in df_fe.columns:
    ing = df_fe['ingreso_total'].dropna()
    p25 = ing.quantile(0.25)
    p75 = ing.quantile(0.75)

    df_fe['bajo_ingreso_relativo'] = (df_fe['ingreso_total'] < p25).astype(int)
    df_fe['alto_ingreso_relativo'] = (df_fe['ingreso_total'] > p75).astype(int)
    df_fe['ingreso_cero']          = (df_fe['ingreso_total'] == 0).astype(int)
    features_created += ['bajo_ingreso_relativo', 'alto_ingreso_relativo', 'ingreso_cero']

    print(f"Percentil 25 ingreso_total : {p25:,.2f}")
    print(f"Percentil 75 ingreso_total : {p75:,.2f}")
    print(f"bajo_ingreso_relativo      : {df_fe['bajo_ingreso_relativo'].sum():,}  ({df_fe['bajo_ingreso_relativo'].mean()*100:.1f}%)")
    print(f"alto_ingreso_relativo      : {df_fe['alto_ingreso_relativo'].sum():,}  ({df_fe['alto_ingreso_relativo'].mean()*100:.1f}%)")
    print(f"ingreso_cero               : {df_fe['ingreso_cero'].sum():,}  ({df_fe['ingreso_cero'].mean()*100:.2f}%)")
else:
    print('ADVERTENCIA: ingreso_total no disponible.')

proporcion_ingreso_principal — media: 0.9754
Percentil 25 ingreso_total : 1,025.00
Percentil 75 ingreso_total : 2,500.00
bajo_ingreso_relativo      : 5,443  (22.6%)
alto_ingreso_relativo      : 5,277  (21.9%)
ingreso_cero               : 0  (0.00%)


---
## 10. Flag de Outliers de Ingreso

Usa los flags de outliers ya calculados en la etapa de preprocesamiento (si existen).

**Feature creado:** `ingreso_outlier`

In [10]:
flag_cols = [c for c in ['INGTOT_outlier_flag', 'INGTRABW_outlier_flag'] if c in df_fe.columns]

if flag_cols:
    flags_df = df_fe[flag_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
    df_fe['ingreso_outlier'] = (flags_df.max(axis=1) == 1).astype(int)
    features_created.append('ingreso_outlier')
    print(f'Columnas usadas: {flag_cols}')
    print(f"ingreso_outlier : {df_fe['ingreso_outlier'].sum():,}  ({df_fe['ingreso_outlier'].mean()*100:.2f}%)")
else:
    # Calcular outliers con IQR si no están disponibles
    if 'ingreso_total' in df_fe.columns:
        ing = df_fe['ingreso_total'].dropna()
        q1, q3 = ing.quantile(0.25), ing.quantile(0.75)
        iqr = q3 - q1
        limite_sup = q3 + 3 * iqr
        df_fe['ingreso_outlier'] = (df_fe['ingreso_total'] > limite_sup).astype(int)
        features_created.append('ingreso_outlier')
        print(f'ingreso_outlier calculado con IQR (límite superior: {limite_sup:,.2f})')
        print(f"ingreso_outlier : {df_fe['ingreso_outlier'].sum():,}  ({df_fe['ingreso_outlier'].mean()*100:.2f}%)")
    else:
        print('ADVERTENCIA: No se pudo calcular ingreso_outlier.')

Columnas usadas: ['INGTOT_outlier_flag', 'INGTRABW_outlier_flag']
ingreso_outlier : 821  (3.41%)


---
## 11. Validación Cruzada con Target

In [11]:
ing_vars = [f for f in ['ingreso_total_log', 'ingreso_laboral_log', 'ingreso_por_hora_log',
                         'bajo_ingreso_relativo', 'alto_ingreso_relativo', 'ingreso_cero']
            if f in df_fe.columns]

if ing_vars:
    print('Correlacion con target_subempleo_horas:')
    corr = df_fe[ing_vars + ['target_subempleo_horas']].corr()['target_subempleo_horas'].drop('target_subempleo_horas')
    display(corr.sort_values(ascending=False).to_frame().rename(columns={'target_subempleo_horas': 'correlacion'}))

Correlacion con target_subempleo_horas:


,correlacion
bajo_ingreso_relativo,0.1180
ingreso_por_hora_log,0.0010
alto_ingreso_relativo,-0.0541
ingreso_total_log,-0.1111
ingreso_laboral_log,-0.1157
ingreso_cero,NaN


---
## 12. Validaciones Finales

In [12]:
assert df_fe.shape[0] == n_original, \
    f'ERROR: el número de filas cambió. Original: {n_original}, actual: {df_fe.shape[0]}'
print(f'Número de filas sin cambios : OK ({n_original:,})')

assert 'target_subempleo_horas' in df_fe.columns
print('target_subempleo_horas presente: OK')

for lv in ['P209H', 'C333', 'C334']:
    assert lv not in df_fe.columns, f'ERROR: {lv} presente.'
print('Variables de leakage ausentes: OK')

print(f'\nFeatures de ingresos creados ({len(features_created)}):')
for feat in features_created:
    nulos = df_fe[feat].isnull().sum()
    print(f'  {feat:<38}: {nulos:>6} nulos  ({nulos/len(df_fe)*100:.2f}%)')

print(f'\nDimensiones finales  : {df_fe.shape[0]:,} filas x {df_fe.shape[1]} columnas')
print(f'Columnas nuevas en este paso: {df_fe.shape[1] - df.shape[1]}')

Número de filas sin cambios : OK (24,054)
target_subempleo_horas presente: OK
Variables de leakage ausentes: OK

Features de ingresos creados (16):
  ingreso_total                         :    774 nulos  (3.22%)
  ingreso_principal                     :    774 nulos  (3.22%)
  ingreso_laboral                       :    774 nulos  (3.22%)
  ingreso_total_log                     :    774 nulos  (3.22%)
  ingreso_principal_log                 :    774 nulos  (3.22%)
  ingreso_laboral_log                   :    774 nulos  (3.22%)
  ingreso_por_hora                      :    774 nulos  (3.22%)
  ingreso_por_hora_log                  :    774 nulos  (3.22%)
  ingreso_secundario_total              :      0 nulos  (0.00%)
  tiene_ingreso_secundario              :      0 nulos  (0.00%)
  proporcion_ingreso_principal          :    774 nulos  (3.22%)
  dependencia_ingreso_principal         :      0 nulos  (0.00%)
  bajo_ingreso_relativo                 :      0 nulos  (0.00%)
  alto_ingreso_relat

---
## 13. Guardar Dataset Final de Feature Engineering

In [13]:
OUTPUT_DIR = Path('../data/feature_engineering')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset final con TODAS las features (demográficas + educativas + laborales + ingresos)
out_main = OUTPUT_DIR / 'epen_features_final.csv'
df_fe.to_csv(out_main, index=False)
print(f'Dataset final guardado  : {out_main}')
print(f'Dimensiones             : {df_fe.shape[0]:,} filas x {df_fe.shape[1]} columnas')

# Reporte de features de ingresos creados
report = pd.DataFrame({
    'feature'   : features_created,
    'tipo_dato' : [str(df_fe[f].dtype) for f in features_created],
    'nulos'     : [df_fe[f].isnull().sum() for f in features_created],
    'n_unicos'  : [df_fe[f].nunique() for f in features_created],
    'pct_nulos' : [round(df_fe[f].isnull().mean() * 100, 4) for f in features_created],
})
out_report = OUTPUT_DIR / 'income_features_created.csv'
report.to_csv(out_report, index=False)
print(f'Reporte guardado        : {out_report}  ({len(report)} features)')

print('\nTodos los archivos guardados correctamente.')
print('Siguiente paso -> train_test_split.ipynb')

Dataset final guardado  : ..\data\feature_engineering\epen_features_final.csv
Dimensiones             : 24,054 filas x 114 columnas
Reporte guardado        : ..\data\feature_engineering\income_features_created.csv  (16 features)

Todos los archivos guardados correctamente.
Siguiente paso -> train_test_split.ipynb
